In [ ]:
import numpy as np
from scipy import integrate
import pandas as pd
import ibis
import os
from swed_17.snow17 import SweDB

Brier Skill Score (BSS)

In [ ]:
SWE_DB = ibis.connect("postgres://?service=swe_db")
SNOW17_DB = SweDB(os.environ.get("S17_DB"))

In [ ]:
zonal_swe = SWE_DB.table("zonal_swe")

In [ ]:
START_DATE = "2020-10-01"
ZONE_ID = 794

In [ ]:
snow_17 = SNOW17_DB.for_zone_forecasted("ANBC2H", from_year=START_DATE[0:4])
snow_17 = snow_17[snow_17.index >= START_DATE]
snow_17 = snow_17.reset_index()
snow_17["Date"] = snow_17["Date"].dt.tz_localize("UTC")
snow_17.rename(columns={"SWE (mm)": "Snow-17"}, inplace=True)

In [ ]:
snow_17.head()

In [ ]:
snow_17[snow_17.isna().any(axis=1)]

In [ ]:
isnobal = (
    zonal_swe.filter(
        [
            zonal_swe.cbrfc_zone_id == ZONE_ID,
            zonal_swe.date.hour() == 00,
        ]
    )
    .order_by(zonal_swe.date)
    .execute()
)

isnobal = isnobal.drop(columns=["aso_swe"])
isnobal.rename(
    columns={
        "date": "Date",
        "zone_name": "Zone Name",
        "isnobal_swe": "iSnobal",
        "snodas_swe": "Snodas",
        "ua_swe": "UA",
        "cu_boulder_swe": "CUB",
    },
    inplace=True,
)

In [ ]:
isnobal.head()

In [ ]:
data = pd.merge(
    snow_17.dropna(subset=["Snow-17"]),
    isnobal,
    on=["Date", "Zone Name"],
    how="inner",
).set_index("Date")

In [ ]:
data[data.isna().any(axis=1)]

### Overlapping Index

$ \eta (A,B)=\int \min [f_{A}(x),f_{B}(x)]\,dx $

### Centroid Formula
$ T_{c}=\int t\cdot y_{norm}(t)dt $

### Mean Absolute Error (MAE)
The average of the absolute differences at each point.

### Dynamic Time Warping (DTW)

In [ ]:
def pairwise_overlap(data_a, data_b):
    x = np.arange(0, len(data_b), 1)

    # Normalize (PDF)
    area_1 = integrate.simpson(data_a, x=x)
    area_2 = integrate.simpson(data_b, x=x)
    y1_norm = data_a / area_1
    y2_norm = data_b / area_2

    # Calculate centroids (mean time)
    y1_centroid = integrate.simpson(x * y1_norm, x=x)
    y2_centroid = integrate.simpson(x * y2_norm, x=x)

    # Overlapping index
    overlapping = integrate.simpson(np.minimum(y1_norm, y2_norm), x=x)
    # Timing shift (Positive show y2 lagging)
    timing_shift = y2_centroid - y1_centroid
    # Relative Magnitude Ratio
    magnitude_ratio = area_1 / area_2
    # Magnitude Difference (Net Area)
    net_diff = area_1 - area_2

    difference = np.abs(data_a - data_b)
    # Mean absolute error
    mae = np.mean(difference)

    return {
        "overlapping": overlapping.round(2),
        "timing_shift": timing_shift.round(2),
        "magnitude": magnitude_ratio.round(2),
        "net": int(net_diff),
        "mae": mae.round(2),
    }

In [ ]:
cols = ["Snow-17", "iSnobal", "Snodas", "UA", "CUB"]
year_stats = {}

In [ ]:
for year in range(2022, 2026):
    year_data = data[
        (data.index >= f"{year - 1}-10-01") & (data.index < f"{year}-10-01")
    ].sort_index()
    overlapping = pd.DataFrame(index=cols, columns=cols).astype(float)
    timing_shift = pd.DataFrame(index=cols, columns=cols).astype(float)
    net_diff = pd.DataFrame(index=cols, columns=cols).astype(float)
    magnitude_diff = pd.DataFrame(index=cols, columns=cols).astype(float)
    mae = pd.DataFrame(index=cols, columns=cols).astype(float)
    for col_a in cols:
        for col_b in cols:
            if col_a == col_b:
                value = {
                    "overlapping": 1,
                    "timing_shift": 1,
                    "net": 1,
                    "magnitude": 1,
                    "mae": 1,
                }
            else:
                try:
                    value = pairwise_overlap(year_data[col_a], year_data[col_b])
                except Exception as e:
                    print(col_a)
                    print(col_b)
                    raise e

            overlapping.loc[col_a, col_b] = value["overlapping"]
            timing_shift.loc[col_a, col_b] = value["timing_shift"]
            net_diff.loc[col_a, col_b] = value["net"]
            magnitude_diff.loc[col_a, col_b] = value["magnitude"]
            mae.loc[col_a, col_b] = value["mae"]

    year_stats[year] = {
        "overlapping": overlapping,
        "timing_shift": timing_shift,
        "magnitude": magnitude_diff,
        "net": net_diff,
        "mae": mae,
    }

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
import plotly.io as pio

pio.renderers.default = "notebook"

https://plotly.com/python/heatmaps/

In [ ]:
def show_year(year, data):
    data = data[year]

    fig = make_subplots(
        rows=1,
        cols=5,
        shared_yaxes=True,
        subplot_titles=[
            "Overlapping",
            "Timing Shift",
            "Magnitude",
            "Total Net (mm)",
            "MAE (mm/day)",
        ],
        horizontal_spacing=0.05,
    )
    column = 1

    figures = []
    for name, stat in data.items():
        # Mask symmetrical values
        mask = np.triu(np.ones(stat.shape), k=0).astype(bool)
        stat = stat.astype("string")
        stat[mask] = ""

        # Subset to reduce "NaN"
        stat = stat.iloc[1:4, 0:3]

        sub_fig = go.Heatmap(
            z=stat.values,
            y=stat.index,
            x=stat.columns,
            text=stat.values,
            texttemplate="%{text}",
            hoverongaps=False,
            coloraxis=f"coloraxis{column}",
            hoverinfo=["x", "y", "z"],
        )
        fig.add_trace(sub_fig, row=1, col=column)
        column += 1
    fig.update_layout(
        height=400,
        width=1400,
        title_text=year,
        coloraxis1=dict(
            colorscale="tempo", colorbar_x=0.16, colorbar_thickness=10
        ),
        coloraxis2=dict(
            colorscale="tealrose_r",
            cmid=1,
            colorbar_x=0.37,
            colorbar_thickness=10,
        ),
        coloraxis3=dict(
            colorscale="tealrose_r",
            cmid=1,
            colorbar_x=0.58,
            colorbar_thickness=10,
        ),
        coloraxis4=dict(
            colorscale="tealrose_r",
            cmid=1,
            colorbar_x=0.79,
            colorbar_thickness=10,
        ),
        coloraxis5=dict(colorscale="amp", colorbar_x=1, colorbar_thickness=10),
    )
    return fig

In [ ]:
show_year(2022, year_stats)

In [ ]:
show_year(2023, year_stats)

In [ ]:
show_year(2024, year_stats)

In [ ]:
show_year(2025, year_stats)

In [ ]:
data[cols].plot()